# Model Comparison: Global Mean → Rate Type → Categoricals → All Features (no VIF)

Four progressively richer models on the same train/val/test split, so each step's RMSE tells you exactly how much that additional information is worth:

1. **Global mean** — dumbest possible baseline
2. **`rate_type_index` only** — residential/commercial/industrial group means
3. **Categorical-only** (`rate_type_index` + `ownership` + `service_type`) — the 8-variable model from before
4. **All features, no VIF** — everything that survives basic cleanup (log transforms, compositional reference-category drops) but *without* the VIF pruning step, so nothing gets silently dropped

Split is by ZIP (`GroupShuffleSplit`), not by row — once real ZIP-level numeric features (income, business, race, regional mix) are in the model, a row-level split risks the same ZIP appearing in both train and test.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf

pd.set_option('display.max_columns', 50)


## 2. Load Data

In [ ]:
df = pd.read_parquet('data/cleaned_data.parquet', engine='pyarrow')
print(f'Shape: {df.shape}')

y = df['rate'].copy()
X = df.drop(columns=['rate']).copy()
zip_groups = df['zip'].copy()   # keep for the group split, before zip gets dropped from X


## 3. Basic Column Fixes (same as the main modeling notebook)

In [ ]:
X = X.drop(columns=['zip'])
X['rate_type_index'] = X['rate_type_index'].astype(str)

utility_cols = ['Utility Res Sales MWh', 'Utility Com Sales MWh', 'Utility Ind Sales MWh']
X[utility_cols] = X[utility_cols].apply(pd.to_numeric, errors='coerce')
for c in utility_cols:
    X[c + '_was_missing'] = X[c].isna().astype(int)


In [ ]:
num_cols_all = X.select_dtypes(include=[np.number]).columns
zero_var_cols = [c for c in num_cols_all if X[c].std() == 0 or X[c].nunique() <= 1]
print('Dropping zero-variance columns:', zero_var_cols)
X = X.drop(columns=zero_var_cols)


In [ ]:
est_cols = [c for c in [
    'est', 'n<5', 'n5_9', 'n10_19', 'n20_49',
    'n50_99', 'n100_249', 'n250_499', 'n500_999', 'n1000',
] if c in X.columns]

zero_inflated_volume_cols = [c for c in utility_cols + ['Utility Total_Generation_MWh'] if c in X.columns]
plain_volume_cols = [c for c in ['total_population', 'households'] if c in X.columns]

for c in zero_inflated_volume_cols:
    X[c + '_is_zero'] = (X[c] == 0).astype(int)

log_candidates = est_cols + zero_inflated_volume_cols + plain_volume_cols
X[log_candidates] = np.log1p(X[log_candidates].clip(lower=0))


## 4. Compositional Reference-Category Drops (kept — this is not VIF)

These fix an actual correctness issue (features that sum to ~100% by construction), not a "which features are useful" judgment call — keeping them in every version of the model, including the no-VIF one, so "all features" doesn't mean "the raw redundant encoding." The income brackets get the same treatment here, which they were missing in the main notebook.

In [ ]:
# Race -> shares of population, drop total + one reference category
race_cols = [c for c in X.columns if c in (
    'white_alone', 'black_or_african_american_alone',
    'american_indian_and_alaska_native_alone', 'asian_alone',
    'native_hawaiian_and_other_pacific_islander_alone',
    'some_other_race_alone', 'two_or_more_races:',
    'two_or_more_races:_two_races_including_some_other_race',
    'two_or_more_races:_two_races_excluding_some_other_race,_and_three_or_more_races',
)]
for c in race_cols:
    X[c] = X[c] / X['total_population'].replace(0, np.nan)
X = X.drop(columns=['total_population', 'some_other_race_alone'])


In [ ]:
# Regional source mix & generation mix -> drop one reference category per block
regional_pct_cols = [c for c in X.columns if 'Regional' in c and 'Non-Base' not in c]
nonbase_pct_cols  = [c for c in X.columns if 'Non-Base Regional' in c]
gen_pct_cols      = [c for c in X.columns if c.startswith('Utility ') and c.endswith('_Pct')]

reference_cols_to_drop = [
    sorted(regional_pct_cols)[0] if regional_pct_cols else None,
    sorted(nonbase_pct_cols)[0] if nonbase_pct_cols else None,
    sorted(gen_pct_cols)[0] if gen_pct_cols else None,
]
reference_cols_to_drop = [c for c in reference_cols_to_drop if c is not None]
print('Dropping as reference categories:', reference_cols_to_drop)
X = X.drop(columns=reference_cols_to_drop)


In [ ]:
# Income brackets -> same compositional issue, same fix (missing from the
# main notebook until now -- this was the actual cause of the VIF cascade)
income_bracket_cols = [c for c in X.columns
                        if c.startswith('households_') and 'median' not in c
                        and 'mean' not in c and 'percent_allocated' not in c]
print('Income bracket columns:', income_bracket_cols)
if income_bracket_cols:
    X = X.drop(columns=[sorted(income_bracket_cols)[0]])


## 5. Train / Val / Test Split — Grouped by ZIP

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=0)
train_idx, temp_idx = next(gss1.split(X, y, groups=zip_groups))
X_train, X_temp = X.iloc[train_idx].copy(), X.iloc[temp_idx].copy()
y_train, y_temp = y.iloc[train_idx].copy(), y.iloc[temp_idx].copy()
zip_temp = zip_groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=0)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups=zip_temp))
X_val, X_test = X_temp.iloc[val_idx].copy(), X_temp.iloc[test_idx].copy()
y_val, y_test = y_temp.iloc[val_idx].copy(), y_temp.iloc[test_idx].copy()

# sanity check: no ZIP should appear in more than one split
train_zips = set(zip_groups.iloc[train_idx])
val_zips   = set(zip_groups.iloc[temp_idx].iloc[val_idx])
test_zips  = set(zip_groups.iloc[temp_idx].iloc[test_idx])
print('ZIP overlap train/val:', len(train_zips & val_zips))
print('ZIP overlap train/test:', len(train_zips & test_zips))
print('ZIP overlap val/test:', len(val_zips & test_zips))

utility_cols_present = [c for c in utility_cols if c in X_train.columns]
utility_medians = X_train[utility_cols_present].median()
for split in (X_train, X_val, X_test):
    split[utility_cols_present] = split[utility_cols_present].fillna(utility_medians)

print('\nTrain:', X_train.shape, ' Val:', X_val.shape, ' Test:', X_test.shape)


## 6. Helper: TF Linear Regression

Default (Glorot) initialization, not `ones_initializer` — with standardized unbounded numeric inputs, starting every weight at 1 can put the initial prediction wildly off scale for models with more than a handful of features.

In [ ]:
def build_model(num_features, learning_rate):
    tf.keras.backend.clear_session()
    tf.random.set_seed(0)

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(1, input_shape=(num_features,))
    ])
    model.compile(
        optimizer=tf.keras.optimizers.legacy.SGD(learning_rate=learning_rate),
        loss='mse',
        metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')],
    )
    return model


def fit_and_eval(preprocessor, feature_cols, learning_rate=0.01, epochs=30, batch_size=2048):
    Xtr = preprocessor.fit_transform(X_train[feature_cols])
    Xv  = preprocessor.transform(X_val[feature_cols])
    Xte = preprocessor.transform(X_test[feature_cols])

    model = build_model(Xtr.shape[1], learning_rate)
    model.fit(Xtr, y_train, epochs=epochs, batch_size=batch_size,
              validation_data=(Xv, y_val), verbose=0)

    preds_val  = model.predict(Xv, verbose=0).flatten()
    preds_test = model.predict(Xte, verbose=0).flatten()

    return {
        'val_rmse':  np.sqrt(mean_squared_error(y_val, preds_val)),
        'val_mae':   mean_absolute_error(y_val, preds_val),
        'test_rmse': np.sqrt(mean_squared_error(y_test, preds_test)),
        'test_mae':  mean_absolute_error(y_test, preds_test),
        'n_features': Xtr.shape[1],
    }


## 7. Model 1 — Global Mean Baseline

In [ ]:
train_mean = y_train.mean()

baseline_global = {
    'val_rmse':  np.sqrt(mean_squared_error(y_val, np.full(len(y_val), train_mean))),
    'val_mae':   mean_absolute_error(y_val, np.full(len(y_val), train_mean)),
    'test_rmse': np.sqrt(mean_squared_error(y_test, np.full(len(y_test), train_mean))),
    'test_mae':  mean_absolute_error(y_test, np.full(len(y_test), train_mean)),
    'n_features': 0,
}
print(baseline_global)


## 8. Model 2 — `rate_type_index` Only

In [ ]:
rate_type_train_means = y_train.groupby(X_train['rate_type_index']).mean()

preds_val_rt  = X_val['rate_type_index'].map(rate_type_train_means)
preds_test_rt = X_test['rate_type_index'].map(rate_type_train_means)

baseline_rate_type = {
    'val_rmse':  np.sqrt(mean_squared_error(y_val, preds_val_rt)),
    'val_mae':   mean_absolute_error(y_val, preds_val_rt),
    'test_rmse': np.sqrt(mean_squared_error(y_test, preds_test_rt)),
    'test_mae':  mean_absolute_error(y_test, preds_test_rt),
    'n_features': 1,
}
print(baseline_rate_type)


## 9. Model 3 — Categorical-Only (`rate_type_index` + `ownership` + `service_type`)

In [ ]:
cat_only_cols = ['rate_type_index', 'ownership', 'service_type']

cat_only_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_only_cols),
])

result_cat_only = fit_and_eval(cat_only_preprocessor, cat_only_cols)
print(result_cat_only)


## 10. Model 4 — All Features, No VIF

Everything left in `X` after Sections 3–4 (log transforms + compositional reference-category drops), with no multicollinearity-based pruning. This is the direct "what if I just throw everything in" comparison.

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
all_feature_cols = cat_cols + num_cols

print(f'Using {len(all_feature_cols)} raw columns ({len(cat_cols)} categorical, {len(num_cols)} numeric)')

all_features_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols),
])

result_all_features = fit_and_eval(all_features_preprocessor, all_feature_cols)
print(result_all_features)


## 11. Comparison

In [ ]:
comparison = pd.DataFrame([
    {'model': '1. Global mean',        **baseline_global},
    {'model': '2. rate_type_index only', **baseline_rate_type},
    {'model': '3. Categorical-only (rate_type + ownership + service_type)', **result_cat_only},
    {'model': '4. All features, no VIF', **result_all_features},
])
comparison['test_rmse_improvement_vs_global'] = (
    1 - comparison['test_rmse'] / baseline_global['test_rmse']
)
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison['model'], comparison['test_rmse'], color='#4878CF')
ax.set_ylabel('Test RMSE ($/kWh)')
ax.set_title('Test RMSE by Model')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


**How to read this:** each row should beat the one above it by a real, explainable margin. If Model 4 barely beats Model 3, the numeric features (income/business/race/regional mix) aren't adding much despite the added complexity. If Model 4 beats Model 3 by a lot *and* the gap between Model 4's train and val loss is small (check with the loss-curve code from earlier in the conversation), that's a real, trustworthy improvement — worth then deciding whether VIF-based pruning on top of this is worth the interpretability trade-off, now that you have a genuine "all features" number to prune from instead of the broken 8-column version.